# IMPORTS

In [ ]:
import numpy as np
import scipy
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import scipy.stats as stats
import warnings
import seaborn as sns
from pathlib import Path
from datetime import datetime
from windrose import WindroseAxes

# diive 
import importlib.metadata
version_diive = importlib.metadata.version("diive")
print(f"diive version: v{version_diive}")
from diive.core.dfun.stats import sstats  # Time series stats
from diive.core.io.files import save_parquet, load_parquet
from diive.core.plotting.timeseries import TimeSeries  # For simple (interactive) time series plotting
from diive.core.plotting.heatmap_datetime import HeatmapDateTime, HeatmapYearMonth

warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)

# Matplotlib defaults
plt.rcParams["figure.autolayout"] = True
plt.rcParams["figure.figsize"] = (10, 4)

# LOAD DATA

In [ ]:
FILEPATH = r"..\\60_USTAR_FILTERING\\62.1_FLUXES_L3.3_NEE_LE_H_FN2O_FCH4_REDDYPROC.parquet"
data_original = load_parquet(filepath=FILEPATH)
maindf = data_original.copy()

maindf

In [ ]:
# Select fluxes
FLUXES = ['FN2O_L3.3_CUT_50_QCF0', 'NEE_L3.3_CUT_50_QCF0', 'FCH4_L3.3_CUT_50_QCF0']
WIND = ['WD', 'WD_SIGMA']
# Plot time series 
ax = maindf[FLUXES + WIND].plot(subplots=True, figsize=(10, 6), x_compat=True)
plt.show()

# WIND ROSES

In [ ]:
# Classify day/night
df = maindf.copy()
df["period"] = df["SW_IN_POT"].apply(lambda x: "Day" if x > 0 else "Night")

# Function to plot windrose
def plot_windrose(data, title):
    ax = WindroseAxes.from_ax()
    ax.bar(
        data["WD"], data["WS"],
        normed=True, opening=0.8, edgecolor="white",
        bins=[0,1,2,3,4],
        cmap=plt.cm.viridis
    )
    ax.set_title(title, fontsize=12)
    ax.set_legend(title="Wind speed (m/s)", loc="lower right", bbox_to_anchor=(1.3, 0))
    return ax

# --- Full data windrose ---
plot_windrose(df, "All data")

# --- Daytime windrose ---
plot_windrose(df[df["period"]=="Day"], "Day (SW_IN_POT > 0)")

# --- Nighttime windrose ---
plot_windrose(df[df["period"]=="Night"], "Night (SW_IN_POT == 0)")

plt.show()


# SPLIT INTO PARCELS

We assume the division line lies between 87° and 267°. A symmetric buffer is used to avoid ambiguous assignment.

In [ ]:
wd  = maindf['WD']
sig = maindf['WD_SIGMA']
buffer = 3.0  # degrees; set to 0 if you don't want a center buffer

# interval edges (handle wrap)
factor = 1  # how many sigma to use for the interval
lo = (wd - (sig * factor)) % 360
hi = (wd + (sig * factor)) % 360
wraps = lo > hi

# --- FULL windows (for interval containment, no buffer) ---
# A_full = [0,87) ∪ (267,360]; B_full = (87,26)
lower_angle = 87
higher_angle = 287
A_full = ((~wraps) & (lo >= 0)   & (hi <= lower_angle)) | \
         ((~wraps) & (lo >= higher_angle) & (hi <= 360)) | \
         (wraps    & (hi <= lower_angle)  & (lo >= higher_angle))
B_full = (~wraps) & (lo >= lower_angle) & (hi <= higher_angle)

# --- SHRUNK windows (for center classification with buffer) ---
center_A_shrunk = ((wd >= 0 + buffer)   & (wd <  lower_angle - buffer)) | \
                  ((wd >  higher_angle + buffer) & (wd <= 360 - buffer))
center_B_shrunk =  (wd >  lower_angle + buffer)  & (wd <  higher_angle - buffer)
center_buffer   = ~(center_A_shrunk | center_B_shrunk) & wd.notna()

# --- certainty logic ---
A_certain   = center_A_shrunk & A_full
A_uncertain = center_A_shrunk & ~A_full

B_certain   = center_B_shrunk & B_full
B_uncertain = center_B_shrunk & ~B_full

# --- labels (init with NaN so missing WD stays NaN) ---
maindf['parcel'] = np.nan
maindf['parcel_certainty'] = np.nan

maindf.loc[center_buffer,     'parcel'] = 'buffer'
maindf.loc[center_A_shrunk,   'parcel'] = 'A'
maindf.loc[center_B_shrunk,   'parcel'] = 'B'

maindf.loc[A_certain | B_certain,   'parcel_certainty'] = 'certain'
maindf.loc[A_uncertain | B_uncertain, 'parcel_certainty'] = 'uncertain'


# Data loss to buffer and uncertainty assessment
for flux in FLUXES:
    total_count = maindf[flux].notna().sum()
    buffer_count = maindf.loc[maindf['parcel'] == 'buffer', flux].notna().sum()
    uncertain_count = maindf.loc[maindf['parcel_certainty'] == 'uncertain', flux].notna().sum()
    perc_loss_buffer   = buffer_count   / total_count * 100 if total_count else np.nan
    perc_loss_uncertain = uncertain_count / total_count * 100 if total_count else np.nan
    print(f"{flux}: {perc_loss_buffer:.2f}% lost to buffer ({buffer}°), "
          f"{perc_loss_uncertain:.2f}% uncertain")

maindf

# CHECK DAY-NIGHT DISTRIBUTION

In [ ]:
def plot_weekly_distribution(df, title_suffix=""):
    # Make sure timestamp is datetime
    df["TIMESTAMP_MIDDLE"] = pd.to_datetime(df["TIMESTAMP_MIDDLE"])

    # Extract year-week
    df["week_start"] = df["TIMESTAMP_MIDDLE"].dt.to_period("W").apply(lambda r: r.start_time)

    # Classify day vs night
    df["period"] = df["SW_IN_POT"].apply(lambda x: "Day" if x > 0 else "Night")

    # Count
    counts = (
        df.groupby(["week_start","period","parcel"])
          .size().reset_index(name="n")
    )

    # Normalize to proportions
    counts["prop"] = counts.groupby(["week_start","period"])["n"].transform(lambda x: x / x.sum())
    counts = counts.sort_values("week_start")

    # Plot
    g = sns.relplot(
        data=counts,
        x="week_start", y="prop", hue="parcel",
        col="period", kind="line",
        height=5, aspect=1.5,
        marker="o"
    )

    g.set_axis_labels("Week", "Proportion")
    g.set_titles("{col_name} " + title_suffix)
    g._legend.set_title("Parcel")
    for ax in g.axes.flat:
        ax.tick_params(axis="x", rotation=45)

    plt.show()


# Full dataset
plot_weekly_distribution(maindf.reset_index().copy(), title_suffix="(all data)")

# Subset: parcel_certainty == 'certain'
df_certain = maindf[maindf['parcel_certainty']=='certain'].reset_index().copy()
plot_weekly_distribution(df_certain, title_suffix="(parcel_certainty = certain)")


# PLOTS

In [ ]:
# --- Period definitions ---
periods = [
    ("Fert 1",  "2024-03-05", "2024-04-09"),
    ("Fert 2",  "2024-04-09", "2024-05-06"),
    ("Fert 3 (on 06.05 in A, on 15.05 in B)", "2024-05-06", "2024-05-30"),
    ("Pre-harvest wheat", "2024-05-30", "2024-07-25"),
    ("Post-harvest wheat", "2024-07-25", "2024-08-22"),
    ("After digestate application ", "2024-08-22", "2024-09-01"),
    ("GL with/out Plantago", '2024-09-01', '2025-06-05')
]

flux = FLUXES[0]
fixed_colors = {'A': 'red', 'B': 'green', 'buffer': 'gray'}

for label, start, end in periods:
    indat = maindf.loc[start:end].copy()
    if indat.empty:
        print(f"No data for period: {label} ({start} to {end})")
        continue

    plt.figure(figsize=(10, 4))
    for parcel in indat['parcel'].dropna().unique():
        for cert, alpha in [('certain', 0.8), ('uncertain', 0.3)]:
            sub = indat[(indat['parcel'] == parcel) &
                        (indat['parcel_certainty'] == cert)]
            if sub.empty or (flux not in sub.columns):
                continue
            plt.scatter(
                sub.index, sub[flux],
                label=f"{parcel} ({cert})",
                s=10,
                alpha=alpha,
                color=fixed_colors.get(parcel, 'black'),
            )

    plt.title(f"{flux} — {label}")
    plt.xlabel('')
    plt.ylabel(flux)
    plt.legend()
    plt.tight_layout()
    plt.show()

# PRELIMINARY COMPARISON TRT

In [ ]:
# Keep only daytime
df_n2o = maindf.copy()

# Build parcel-specific series for FN2O flux comparison
df_n2o = df_n2o[['FN2O_L3.3_CUT_50_QCF', 'parcel', 'parcel_certainty']].copy()
df_n2o_certain = df_n2o.copy()
df_n2o['n2o_flux_A'] = np.nan
df_n2o['n2o_flux_B'] = np.nan
mask_A = df_n2o['parcel'] == 'A'
mask_A_certain   = mask_A & (df_n2o_certain['parcel_certainty'] == 'certain')
mask_B = df_n2o['parcel'] == 'B'
mask_B_certain   = mask_B & (df_n2o_certain['parcel_certainty'] == 'certain')
df_n2o.loc[mask_A, 'n2o_flux_A'] = df_n2o.loc[mask_A, 'FN2O_L3.3_CUT_50_QCF']
df_n2o.loc[mask_B, 'n2o_flux_B'] = df_n2o.loc[mask_B, 'FN2O_L3.3_CUT_50_QCF']
df_n2o_certain.loc[mask_A, 'n2o_flux_A'] = df_n2o_certain.loc[mask_A_certain, 'FN2O_L3.3_CUT_50_QCF']
df_n2o_certain.loc[mask_B, 'n2o_flux_B'] = df_n2o_certain.loc[mask_B_certain, 'FN2O_L3.3_CUT_50_QCF']
df_n2o = df_n2o.drop(columns=['parcel', 'parcel_certainty', 'FN2O_L3.3_CUT_50_QCF'])
df_n2o_certain = df_n2o_certain.drop(columns=['parcel', 'parcel_certainty', 'FN2O_L3.3_CUT_50_QCF'])

# Resample (specify the period and the aggregation method)
sampling_period = '6h'
resampled_df = df_n2o.resample(sampling_period).mean()
resampled_df_certain = df_n2o_certain.resample(sampling_period).mean()

# Difference (A - B)
resampled_df['flux_diff_A-B'] = resampled_df['n2o_flux_A'] - resampled_df['n2o_flux_B']
resampled_df_certain['flux_diff_A-B'] = resampled_df_certain['n2o_flux_A'] - resampled_df_certain['n2o_flux_B']

# Ensure pairs: drop rows where either A or B is NaN
resampled_df = resampled_df.dropna(how='any')
resampled_df_certain = resampled_df_certain.dropna(how='any')

periods = [
    ("2023-11-01", "2024-07-25"),  # Wheat
    ("2024-10-01", "2025-06-05"),  # Grassland
]

for start, end in periods:
    for df, title in [
        (resampled_df, "All data"),
        (resampled_df_certain, "Certain only"),
    ]:
        # Subset by date range
        subdf = df.loc[start:end]

        a = subdf['n2o_flux_A']
        b = subdf['n2o_flux_B']
        x = subdf.index

        eps = 0.1  # detection limit
        pos = a > b + eps   # A > B
        neg = b > a + eps   # B > A

        plt.figure(figsize=(10, 6))
        # option 1: scatter + lines
        plt.scatter(x, a, label='standard fertilization (A)', c='red', s=10, alpha=0.6)
        plt.scatter(x, b, label='precision fertilization (B)', c='green', s=10, alpha=0.6)
        plt.vlines(x[pos], ymin=b[pos], ymax=a[pos], colors='tab:red', alpha=1, linewidth=1.5, label='A > B')
        plt.vlines(x[neg], ymin=a[neg], ymax=b[neg], colors='tab:green', alpha=1, linewidth=1.5, label='B > A')

        # option 2: weekly means
        # a_weekly = a.resample('W').mean()
        # b_weekly = b.resample('W').mean()
        # plt.plot(a_weekly.index, a_weekly, label='standard fertilization (A)', c='red')
        # plt.plot(b_weekly.index, b_weekly, label='precision fertilization (B)', c='green')

        plt.title(f"N$_2$O flux comparison (Parcel A vs B) — {title}\n{start} to {end}")
        plt.ylabel('N$_{2}$O Flux (nmol m$^{-2}$ s$^{-1}$)')
        plt.legend()
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## Cumulative sums (A vs B)

In [ ]:
periods = [
    ("2024-03-01", "2024-06-15"),  # Wheat
    ("2024-10-01", "2025-06-05"),  # Grassland
]

# Define labels and colors for the management events
# Colors for the 2 parcels
COLORS = {'A': "#D55E00", 'B': "#0072B2"}
event_lines = [
    {'date': '2024-03-05', 'color': 'black', 'linestyle': '--', 'label': 'FERT\nA: 30 kgN/ha\nB: 32 kgN/ha'},
    {'date': '2024-04-09', 'color': 'black', 'linestyle': '--', 'label': 'FERT\nA: 56 kgN/ha\nB: 80 kgN/ha'},
    {'date': '2024-05-06', 'color': COLORS['A'], 'linestyle': '--', 'label': 'FERT\nA: 60 kgN/ha'},
    {'date': '2024-05-15', 'color': COLORS['B'], 'linestyle': '--', 'label': 'FERT\nB: 26 kgN/ha'}
]

for start, end in periods:
    start_ts = pd.to_datetime(start)
    end_ts   = pd.to_datetime(end)

    for df, title in [
        (resampled_df, "all"),
        (resampled_df_certain, "certain"),
    ]:
        # Subset
        subdf = df.loc[start_ts:end_ts].copy()
        if subdf.empty:
            continue

        # Cumulative (use subset index)
        cum_df = pd.DataFrame(index=subdf.index)
        cum_df['n2o_flux_A'] = subdf['n2o_flux_A'].fillna(0).cumsum()
        cum_df['n2o_flux_B'] = subdf['n2o_flux_B'].fillna(0).cumsum()

        # Plot
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(cum_df.index, cum_df['n2o_flux_A'], label='standard fertilization (A)', color=COLORS['A'])
        ax.plot(cum_df.index, cum_df['n2o_flux_B'], label='precision fertilization (B)', color=COLORS['B'])

        # Annotate only events within the period
        for event in event_lines:
            event_date = pd.to_datetime(event['date'])
            if not (start_ts <= event_date <= end_ts):
                continue
            ax.annotate(
                text=event['label'],
                xy=(mdates.date2num(event_date), 0),
                xytext=(mdates.date2num(event_date), 0.85),
                xycoords=('data', 'axes fraction'),
                textcoords=('data', 'axes fraction'),
                ha='center', va='center',
                rotation=90, rotation_mode='anchor',
                color='black',
                arrowprops=dict(
                    arrowstyle='-|>',
                    color=event['color'],
                    linestyle=event['linestyle'],
                    lw=1.5
                ),
                annotation_clip=False
            )

        # X-axis formatting
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

        ax.set_title(f"Cumulative N$_2$O flux — {title}\n{start} to {end}")
        ax.set_ylabel('Cumulative N$_{2}$O flux (nmol m$^{-2}$)')
        ax.legend(frameon=False)
        fig.tight_layout()
        fig.savefig(f'cumulative_FN2O_{title}_from{start}_to{end}.png', dpi=300, bbox_inches='tight')
        plt.show()

## Difference series and statistics

In [ ]:
# Relative difference time series
for start, end in periods:
    for df, title in [
        (resampled_df, "All data"),
        (resampled_df_certain, "Certain only"),
    ]:
        subdf = df[start:end] 
        ax = subdf['flux_diff_A-B'].plot(kind='line', figsize=(10, 4), x_compat=True)
        ax.axhline(0, color='black', linewidth=0.8)  # Add zero line
        ax.set_title(f"N$_2$O flux difference (A - B) — {title}\n{start} to {end}")
        ax.set_ylabel('A - B (nmol m$^{-2}$ s$^{-1}$)')
        ax.set_xlabel('')
        plt.show()

        # Average difference
        print(f"--- {title} ---\n{start} to {end}")
        avg_diff = subdf['flux_diff_A-B'].mean()
        print(f"Average difference (A - B): {avg_diff:.3f} nmol m^-2 s^-1")

        # Paired t-test
        t_stat, p_value = stats.ttest_rel(subdf['n2o_flux_A'], subdf['n2o_flux_B'])
        print(f"T-statistic: {t_stat:.3f}, p-value: {p_value:.3g}")

# EXPORT

In [ ]:
newcols = [c for c in maindf.columns if c not in data_original.columns]
print("NEW VARIABLES ADDED TO THE DATASET:")
for c in newcols:
    print(f"+ {c}")

In [ ]:
filename = "71.1_FLUXES_L3.3_NEE_LE_H_FN2O_FCH4_REDDYPROC_PARCELS"
maindf.to_csv(f"{filename}.csv", index=True)
save_parquet(data=maindf, filename=filename)

# End of notebook

In [ ]:
dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Finished. {dt_string}")